# Clipt — Roboflow Jersey Detection Model Training

This notebook trains 4 custom YOLOv8n models from Roboflow Universe datasets.

## Prerequisites
- **Runtime: T4 GPU** (Runtime → Change runtime type → T4 GPU)
- Training takes ~2 hours total on Colab free tier

## After Training
Download all 4 `.pt` files and place them in:
```
jersey-detection/app/model/
```
Then commit and push to GitHub:
```bash
git add app/model/*.pt
git commit -m "Add Roboflow trained detection models"
git push
```
Railway will auto-redeploy.

## Section 1 — Setup

In [ ]:
import os
os.environ["ROBOFLOW_API_KEY"] = os.environ.get("ROBOFLOW_API_KEY", "")

!pip install roboflow ultralytics -q
from roboflow import Roboflow
from ultralytics import YOLO

# Uses ROBOFLOW_API_KEY env var — set in Colab via Secrets (key icon in sidebar)
rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])

# Verify GPU
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Section 2 — Download Datasets

Run each cell below. These download 4 datasets from Roboflow Universe.

In [ ]:
# Dataset A — Football digit detector (PRIMARY for football)
# 13,815 images of jersey digits
project_a = rf.workspace("footballplayertracking").project("jerseynumberdetectordigitdetector")
dataset_a = project_a.version(1).download("yolov8")
print(f"Dataset A location: {dataset_a.location}")

In [ ]:
# Dataset B — Football player detector
# Detects players, goalkeepers, refs, ball
project_b = rf.workspace("roboflow-jvuqo").project("football-players-detection-3zvbc")
dataset_b = project_b.version(15).download("yolov8")
print(f"Dataset B location: {dataset_b.location}")

In [ ]:
# Dataset C — Basketball jersey OCR (FIXED — replaced failed Basketball-Players-1)
# Roboflow official: 3,615 NBA Playoffs jersey number images
# Previous dataset produced mAP50: 0.10 — this one should be much better
project_c = rf.workspace("roboflow-jvuqo").project("basketball-jersey-numbers-ocr")
dataset_c = project_c.version(5).download("yolov8")
print(f"Dataset C location: {dataset_c.location}")

In [ ]:
# Dataset D — Football jersey tracker
# Jersey tracking across frames
project_d = rf.workspace("football-tracking").project("football-jersey-tracker")
dataset_d = project_d.version(1).download("yolov8")
print(f"Dataset D location: {dataset_d.location}")

## Section 3 — Train All 4 Models

Each model trains from a YOLOv8n base. Training takes ~20-40 min per model on T4.

In [ ]:
# Model A — Football digit detector (largest dataset, most augmentation)
print("=" * 60)
print("TRAINING MODEL A: Football Digit Detector")
print("=" * 60)

model_a = YOLO("yolov8n.pt")
model_a.train(
    data=f"{dataset_a.location}/data.yaml",
    epochs=60,
    imgsz=640,
    batch=16,
    name="football_digit_detector",
    patience=15,
    device=0,
    augment=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    scale=0.5,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
)
print("Model A training complete!")

In [ ]:
# Model B — Football player detector
print("=" * 60)
print("TRAINING MODEL B: Football Player Detector")
print("=" * 60)

model_b = YOLO("yolov8n.pt")
model_b.train(
    data=f"{dataset_b.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name="football_player_detector",
    patience=10,
    device=0,
)
print("Model B training complete!")

In [ ]:
# Model C — Basketball jersey OCR
print("=" * 60)
print("TRAINING MODEL C: Basketball Jersey OCR")
print("=" * 60)

model_c = YOLO("yolov8n.pt")
model_c.train(
    data=f"{dataset_c.location}/data.yaml",
    epochs=60,
    imgsz=640,
    batch=16,
    name="basketball_jersey_ocr",
    patience=15,
    device=0,
)
print("Model C training complete!")

In [ ]:
# Model D — Football jersey tracker
print("=" * 60)
print("TRAINING MODEL D: Football Jersey Tracker")
print("=" * 60)

model_d = YOLO("yolov8n.pt")
model_d.train(
    data=f"{dataset_d.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name="football_jersey_tracker",
    patience=10,
    device=0,
)
print("Model D training complete!")

## Section 4 — Validate All Models

In [ ]:
print("\n" + "=" * 60)
print("VALIDATION RESULTS")
print("=" * 60)

models = {
    "Football Digits": "runs/detect/football_digit_detector/weights/best.pt",
    "Football Players": "runs/detect/football_player_detector/weights/best.pt",
    "Basketball Jersey OCR": "runs/detect/basketball_jersey_ocr/weights/best.pt",
    "Football Jersey Tracker": "runs/detect/football_jersey_tracker/weights/best.pt",
}

for name, path in models.items():
    if os.path.exists(path):
        model = YOLO(path)
        metrics = model.val()
        print(f"  {name} — mAP50: {metrics.box.map50:.3f}, mAP50-95: {metrics.box.map:.3f}")
    else:
        print(f"  {name} — MISSING (training may have failed)")

print("\nExpected: mAP50 > 0.7 for digit detection, > 0.85 for player detection")

## Section 5 — Download All Weight Files

Run this cell to download all 4 `.pt` files to your computer.

In [ ]:
from google.colab import files
import shutil

weights = {
    "football_digit_detector.pt": "runs/detect/football_digit_detector/weights/best.pt",
    "football_player_detector.pt": "runs/detect/football_player_detector/weights/best.pt",
    "basketball_jersey_ocr.pt": "runs/detect/basketball_jersey_ocr/weights/best.pt",
    "football_jersey_tracker.pt": "runs/detect/football_jersey_tracker/weights/best.pt",
}

print("Downloading weight files...\n")
for output_name, path in weights.items():
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        shutil.copy(path, output_name)
        files.download(output_name)
        print(f"  Downloaded: {output_name} ({size_mb:.1f} MB)")
    else:
        print(f"  MISSING: {path} — re-run training cell above")

print("\n" + "=" * 60)
print("DONE! Place all .pt files in: jersey-detection/app/model/")
print("Then: git add app/model/*.pt && git commit -m 'Add Roboflow models' && git push")
print("=" * 60)